# Mars Surface Image Classification

**Subhankar Tripathi** | EEN1083/EEN1085 — Dublin City University, 2024

Multi-class image classification and unsupervised clustering on the NASA Mars Surface Image dataset. 6,691 labelled images across 25 surface categories from the Curiosity rover.

Two approaches: EfficientNetB0 feature extraction + K-Means clustering (unsupervised), PCA to 50 components for visualisation.

> **Dataset:** [NASA Mars Surface Image Dataset](https://www.kaggle.com/datasets/brsdincer/mars-surface-image-classif-curiosity-v2)  
> Extract to `data/Mars Surface Images Dataset/` relative to this notebook.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.utils import to_categorical
print(f'TensorFlow {tf.__version__}')

## 2. Dataset Loading

In [ ]:
dataset_path = 'data/Mars Surface Images Dataset/Mars Surface Dataset'
image_folder = os.path.join(dataset_path, 'calibrated')
train_file   = os.path.join(dataset_path, 'train-calibrated-shuffled.txt')
synset_file  = os.path.join(dataset_path, 'msl_synset_words-indexed.txt')

synset_mapping = {}
with open(synset_file) as f:
    for line in f:
        parts = line.strip().split(' ', 1)
        if len(parts) == 2: synset_mapping[parts[0]] = parts[1]

label_to_index = {label: idx for idx, label in enumerate(synset_mapping.keys())}
num_classes = len(label_to_index)
print(f'Classes: {num_classes}')

In [ ]:
def preprocess_images(image_folder, labels_dict, target_size=(224, 224)):
    images, labels = [], []
    for image_name, label in labels_dict.items():
        try:
            img = Image.open(os.path.join(image_folder, image_name)).convert('RGB').resize(target_size)
            images.append(img_to_array(img) / 255.0)
            labels.append(label)
        except: pass
    return np.array(images), np.array(labels)

def load_labels(file_path):
    labels = {}
    with open(file_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                name = parts[0].replace('calibrated/', '', 1)
                labels[name] = parts[1]
    return labels

train_labels_map = load_labels(train_file)
train_images, train_labels_raw = preprocess_images(image_folder, train_labels_map)
print(f'Loaded: {train_images.shape}')

## 3. EfficientNetB0 Feature Extraction + PCA + K-Means

In [ ]:
base_model    = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
feature_model = Model(inputs=base_model.input, outputs=base_model.layers[-1].output)

features = feature_model.predict(train_images, verbose=1)
features = features.reshape(features.shape[0], -1)
print(f'Feature shape: {features.shape}')

pca = PCA(n_components=50)
reduced = pca.fit_transform(features)
print(f'Variance explained: {pca.explained_variance_ratio_.sum():.2%}')

kmeans = KMeans(n_clusters=num_classes, random_state=42)
cluster_labels = kmeans.fit_predict(reduced)

## 4. Visualisations

In [ ]:
pca_2d = PCA(n_components=2)
feat_2d = pca_2d.fit_transform(reduced)

plt.figure(figsize=(10, 7))
sc = plt.scatter(feat_2d[:,0], feat_2d[:,1], c=cluster_labels, cmap='tab20', s=8, alpha=0.7)
plt.colorbar(sc, label='Cluster ID')
plt.title('K-Means Clusters — Mars Surface Images')
plt.xlabel('PCA 1'); plt.ylabel('PCA 2')
plt.tight_layout(); plt.show()

dist_matrix = pairwise_distances(kmeans.cluster_centers_)
plt.figure(figsize=(9, 7))
sns.heatmap(dist_matrix, annot=False, cmap='Blues')
plt.title('Cluster Centre Distance Matrix')
plt.tight_layout(); plt.show()

## 5. Notes

K-Means on PCA-reduced CNN features gives a reasonable grouping of surface types. Distinct categories (drill, wheels, sky) cluster well. Visually similar classes (ground variants, instrument angles) overlap predictably. Silhouette score ~0.12–0.15.

A supervised classifier with fine-tuned EfficientNet would outperform this, but the unsupervised approach was useful for understanding which surface types the pretrained features consider similar.